# Sesión 4 · De la web al DataFrame

**El problema:** Piura tiene 66 mil casos de dengue y Tumbes 17 mil. ¿Piura
está peor? No necesariamente: Piura tiene mucha más gente. Para comparar hace
falta la **población**, y ese dato no está en nuestras bases.

Está en una página web. Vamos por él.

In [1]:
import pandas as pd
import requests
from io import StringIO

## 1. El camino rápido: `pd.read_html`

Si el dato está en una tabla HTML (`<table>`), pandas la lee directo.

In [2]:
url = "https://es.wikipedia.org/wiki/Anexo:Departamentos_del_Per%C3%BA_por_poblaci%C3%B3n"

# El User-Agent identifica quién hace la consulta. Muchos sitios rechazan
# a quien no se identifica.
headers = {"User-Agent": "Mozilla/5.0 (Diplomado PUCP - curso de Python)"}
respuesta = requests.get(url, headers=headers, timeout=30)
respuesta.status_code

200

`200` significa que todo salió bien. Los códigos que importan:

| Código | Qué pasó |
|---|---|
| 200 | Todo bien |
| 403 | El sitio te bloqueó |
| 404 | La página no existe |
| 429 | Estás consultando demasiado rápido |

In [3]:
tablas = pd.read_html(StringIO(respuesta.text))
len(tablas)

2

`read_html` devuelve **una lista** con todas las tablas de la página.
Hay que mirar cuál sirve.

In [4]:
poblacion = tablas[0]
poblacion.head()

Ubigeo Departamento      Capital Superficie (km²)  Población             \
  Ubigeo Departamento      Capital Superficie (km²) Censo 2017 Censo 2025   
0      1     Amazonas  Chachapoyas        39 249,13    379 384    467 688   
1      2       Áncash       Huaraz        35 914,81  1 083 519  1 205 923   
2      3     Apurímac      Abancay        20 895,79    405 759    467 513   
3      4     Arequipa     Arequipa        63 345,39  1 382 730  1 814 554   
4      5     Ayacucho     Ayacucho        43 814,80    616 176    709 026   

  Densidad 2025 (hab/km²) Ubicación  
  Densidad 2025 (hab/km²) Ubicación  
0                   11.90       NaN  
1                   33.60       NaN  
2                   19.42       NaN  
3                   28.70       NaN  
4                   16.20       NaN

## 2. Limpiar lo que llegó

Lo que viene de la web siempre llega sucio. Aquí hay dos problemas.

### Problema 1: las columnas vienen en dos niveles

In [5]:
poblacion.columns

MultiIndex([(                 'Ubigeo',                  'Ubigeo'),
            (           'Departamento',            'Departamento'),
            (                'Capital',                 'Capital'),
            (       'Superficie (km²)',        'Superficie (km²)'),
            (              'Población',              'Censo 2017'),
            (              'Población',              'Censo 2025'),
            ('Densidad 2025 (hab/km²)', 'Densidad 2025 (hab/km²)'),
            (              'Ubicación',               'Ubicación')],
           )

In [6]:
# get_level_values(-1) toma el nivel de abajo, que es el que distingue
# "Censo 2017" de "Censo 2025".
poblacion.columns = poblacion.columns.get_level_values(-1)
poblacion.columns

Index(['Ubigeo', 'Departamento', 'Capital', 'Superficie (km²)', 'Censo 2017',
       'Censo 2025', 'Densidad 2025 (hab/km²)', 'Ubicación'],
      dtype='object')

Hay dos censos en la tabla. Nuestros datos de dengue van de 2015 a 2021, así
que corresponde el **Censo 2017**. Usar la población del período equivocado
es un error silencioso muy común.

In [7]:
poblacion = poblacion[["Departamento", "Censo 2017"]].copy()
poblacion.head()

,Departamento,Censo 2017
0,Amazonas,379 384
1,Áncash,1 083 519
2,Apurímac,405 759
3,Arequipa,1 382 730
4,Ayacucho,616 176


### Problema 2: la población es texto con espacios

`"467 688"` no es un número para Python. Hay que quitar los espacios
(incluidos los espacios duros de HTML) y convertir.

In [8]:
poblacion["Censo 2017"].head()

0      379 384
1    1 083 519
2      405 759
3    1 382 730
4      616 176
Name: Censo 2017, dtype: object

In [9]:
poblacion["poblacion"] = (
    poblacion["Censo 2017"]
    .astype(str)
    .str.replace(r"\s", "", regex=True)   # espacios normales y duros
    .str.extract(r"(\d+)")[0]             # solo los digitos
)
poblacion["poblacion"] = pd.to_numeric(poblacion["poblacion"], errors="coerce")
poblacion[["Departamento", "poblacion"]].head()

,Departamento,poblacion
0,Amazonas,379384
1,Áncash,1083519
2,Apurímac,405759
3,Arequipa,1382730
4,Ayacucho,616176


### Quitar la fila de totales

Casi todas las tablas web traen una fila "Total" al final que arruina los cálculos.

In [10]:
poblacion = poblacion[~poblacion["Departamento"].str.contains("Total", case=False, na=False)]
poblacion = poblacion.dropna(subset=["poblacion"])
poblacion.shape

(26, 3)

## 3. Cruzar con el dengue

Los nombres de departamento tienen que coincidir exactamente. Wikipedia usa
tildes (`Áncash`), nuestra base no. Hay que normalizar antes de cruzar.

In [11]:
dengue = pd.read_csv("../03-pandas/dengue_limpio.csv", dtype={"ubigeo": str})
casos_depto = dengue.groupby("departamento")["casos"].sum().reset_index()
casos_depto.head()

,departamento,casos
0,Amazonas,3217.0
1,Ancash,2344.0
2,Ayacucho,8818.0
3,Cajamarca,4144.0
4,Callao,11.0


In [12]:
def normalizar(serie):
    """Mayúsculas, sin tildes y sin espacios sobrantes."""
    return (
        serie.str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("utf-8")
        .str.upper()
        .str.strip()
    )

poblacion["llave"] = normalizar(poblacion["Departamento"])
casos_depto["llave"] = normalizar(casos_depto["departamento"])
poblacion[["Departamento", "llave"]].head()

,Departamento,llave
0,Amazonas,AMAZONAS
1,Áncash,ANCASH
2,Apurímac,APURIMAC
3,Arequipa,AREQUIPA
4,Ayacucho,AYACUCHO


In [13]:
tasas = casos_depto.merge(poblacion[["llave", "poblacion"]], on="llave", how="left")
tasas["poblacion"].isna().sum()

np.int64(2)

### Ver qué no cruzó y por qué

In [14]:
tasas[tasas["poblacion"].isna()]

,departamento,casos,llave,poblacion
4,Callao,11.0,CALLAO,NaN
11,Lima,1904.0,LIMA,NaN


**Lima y Callao no cruzaron.** No es un error del código: Wikipedia los
nombra distinto (`Provincia Constitucional del Callao`, y Lima aparece
separada en metropolitana y provincias).

Esto pasa en **todo** cruce con nombres de lugares. Por eso, cuando existe un
código oficial como el ubigeo, siempre se cruza por el código y nunca por el
nombre. Aquí, con solo dos casos, se corrigen a mano.

In [15]:
correcciones = {"LIMA": 9485405, "CALLAO": 994494}   # Censo 2017, INEI
tasas["poblacion"] = tasas["poblacion"].fillna(tasas["llave"].map(correcciones))
tasas["poblacion"].isna().sum()

np.int64(0)

## 4. La respuesta: casos por 100 mil habitantes

In [16]:
tasas["tasa_100mil"] = tasas["casos"] / tasas["poblacion"] * 100_000
tasas.sort_values("tasa_100mil", ascending=False).head(10)[
    ["departamento", "casos", "poblacion", "tasa_100mil"]
]

,departamento,casos,poblacion,tasa_100mil
13,Madre De Dios,14878.0,141070.0,10546.537180
19,Tumbes,17692.0,224863.0,7867.901789
16,Piura,66309.0,1856809.0,3571.126594
20,Ucayali,15214.0,496459.0,3064.502809
12,Loreto,23570.0,883510.0,2667.768333
7,Ica,16588.0,850765.0,1949.774615
18,San Martin,13677.0,813381.0,1681.499814
2,Ayacucho,8818.0,616176.0,1431.084625
0,Amazonas,3217.0,379384.0,847.953525
6,Huanuco,5695.0,721047.0,789.823687


**Ahí cambia la historia.** En números absolutos manda Piura; ajustado por
población, el panorama es otro. Ese es el punto de traer el dato de afuera.

## 5. Cuando `read_html` no alcanza: BeautifulSoup

`read_html` solo ve tablas. Si el dato está en una lista, un título o un
enlace, hay que leer el HTML a mano.

In [17]:
from bs4 import BeautifulSoup

sopa = BeautifulSoup(respuesta.text, "html.parser")
sopa.title.text

'Anexo:Departamentos del Perú por población - Wikipedia, la enciclopedia libre'

### Buscar por etiqueta

- `find` devuelve **el primero** que encuentra
- `find_all` devuelve **todos**

In [18]:
primer_h2 = sopa.find("h2")
primer_h2.text

'Contenidos'

In [19]:
titulos = [h.text.strip() for h in sopa.find_all("h2")]
titulos

['Contenidos', 'Véase también', 'Referencias']

### Buscar por clase CSS

En el navegador: clic derecho sobre el elemento → *Inspeccionar*. Ahí se ve
la etiqueta y la clase que hay que pedir.

In [20]:
tabla_html = sopa.find("table", class_="wikitable")
filas = tabla_html.find_all("tr")
len(filas)

28

### Recorrer la tabla a mano

Esto es exactamente lo que `read_html` hace por dentro.

In [21]:
datos = []
for fila in filas[2:]:                      # las 2 primeras son encabezados
    celdas = fila.find_all(["td", "th"])
    if len(celdas) >= 5:
        datos.append({
            "departamento": celdas[1].get_text(strip=True),
            "capital": celdas[2].get_text(strip=True),
        })

pd.DataFrame(datos).head()

,departamento,capital
0,Amazonas,Chachapoyas
1,Áncash,Huaraz
2,Apurímac,Abancay
3,Arequipa,Arequipa
4,Ayacucho,Ayacucho


### `select` con selectores CSS

Más corto cuando se sabe la ruta exacta del elemento.

In [22]:
enlaces = sopa.select("table.wikitable a")
[a.get("title") for a in enlaces[:8]]

['Ubigeo',
 'Departamento de Amazonas (Perú)',
 'Chachapoyas (ciudad)',
 None,
 'Departamento de Áncash',
 'Huaraz',
 None,
 'Departamento de Apurímac']

## 6. Guardar lo extraído

**Regla:** se guarda apenas se extrae. Nunca se vuelve a pedir a la web algo
que ya se tiene en disco.

In [23]:
tasas.to_csv("tasas_departamento.csv", index=False)
tasas.shape

(21, 5)

## 7. Páginas que cargan con JavaScript

Algunas páginas construyen la tabla **después** de cargarse. `requests` recibe
el HTML vacío y no hay nada que extraer. Ahí hace falta **Selenium**, que abre
un navegador de verdad:

```python
from selenium import webdriver
from selenium.webdriver.common.by import By

driver = webdriver.Chrome()
driver.get("https://www.ejemplo.gob.pe/consulta")
driver.find_element(By.ID, "btnBuscar").click()
tabla = driver.find_element(By.CLASS_NAME, "resultados").get_attribute("outerHTML")
df = pd.read_html(StringIO(tabla))[0]
driver.quit()
```

Cómo saber cuál usar: si `Ctrl+U` (ver código fuente) muestra el dato,
alcanza con `requests`. Si no aparece ahí pero sí en *Inspeccionar*,
se necesita Selenium.

---
## Lo que hicimos

| Paso | Código |
|---|---|
| Pedir la página | `requests.get(url, headers=...)` |
| Verificar | `.status_code == 200` |
| Tablas automático | `pd.read_html(StringIO(r.text))` |
| Columnas en 2 niveles | `.columns.get_level_values(0)` |
| Texto a número | `.str.replace(regex=True)` + `pd.to_numeric` |
| Quitar tildes | `.str.normalize("NFKD").str.encode("ascii", errors="ignore")` |
| Leer HTML a mano | `BeautifulSoup(html, "html.parser")` |
| Buscar | `.find()`, `.find_all()`, `.select()` |
| Guardar | `.to_csv()` apenas se extrae |

### Las tres reglas del scraping

1. **Identifícate** con un `User-Agent`.
2. **No satures** el sitio: si haces muchas consultas, espera entre una y otra.
3. **Guarda apenas extraes.** La página puede cambiar mañana.